# Week 8 · Topic 3 — Light Regex (optional extra)

**Mohammed Mohammed Alhaji**  
CST1510 · 22 November 2026

**Box we are building: INPUT, matched by shape**

Week 7's string methods need to know the exact text. This week: finding
something by its *shape* instead.

> **Already know this?** Skip to the *Topic 3 Challenge*.

> **Optional extra.** You do not need this topic for the Week 8 lab. Come back to it when you meet messy text that has no separator to split on.

---
## 1. Why Week 7's tools are not enough here

"Does this contain an IP address" is not "does this contain one exact
substring" — the IP is different every time.

In [1]:
import re

line = "2026-09-08 connection from 10.0.0.5 refused"

match = re.search(r"\d+\.\d+\.\d+\.\d+", line)
print(match.group())

10.0.0.5


### Your turn
Given a line containing a different IP address, use the same pattern to find it.

In [2]:
other_line = "2026-11-22 connection from 172.16.0.8 accepted"
match = re.search(r"\d+\.\d+\.\d+\.\d+", other_line)
if match:
    print(match.group())


172.16.0.8


---
## 2. `re.search()` returns `None` when nothing matches

Same shape as Week 7's `.find()` returning `-1` — check before you use it.

In [3]:
no_ip = "no address in this line"
match = re.search(r"\d+\.\d+\.\d+\.\d+", no_ip)
print(match)

if match:
    print(match.group())
else:
    print("no IP found")

None
no IP found


### Your turn
Run the search on a line you know has no match, and check `match` before calling `.group()`.

In [4]:
message = "The server started successfully"
match = re.search(r"\d+\.\d+\.\d+\.\d+", message)
if match:
    print(match.group())
else:
    print("no IP found")


no IP found


---
## 3. `re.findall()` — every match, as a list

In [5]:
log = "attempts from 10.0.0.5 and 192.168.1.1 and 10.0.0.9"
all_ips = re.findall(r"\d+\.\d+\.\d+\.\d+", log)
print(all_ips)

['10.0.0.5', '192.168.1.1', '10.0.0.9']


### Your turn
Use `re.findall()` to pull every 4-digit year out of a string containing several dates.

In [6]:
dates = "Created 2024-04-10, updated 2025-09-18, reviewed 2026-11-22"
years = re.findall(r"\b\d{4}\b", dates)
print(years)


['2024', '2025', '2026']


---
## 4. A second pattern, and using it on real pulled data

In [7]:
date_match = re.search(r"\d{4}-\d{2}-\d{2}", line)
print(date_match.group())

sample = "Supports Python 3.12 and above, tested on 3.10 too"
print(re.findall(r"\d+\.\d+", sample))

2026-09-08
['3.12', '3.10']


### Your turn
Write a pattern that matches something shaped like `something@something.something` (it does not need to be perfect).

In [8]:
message = "Contact help@example.com or mohammed@student.ac.uk"
email_pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
print(re.findall(email_pattern, message))


['help@example.com', 'mohammed@student.ac.uk']


---
## 5. Capture groups — pulling out the pieces, not just the whole match

Parentheses `(...)` in a pattern mark a "capture group" — a part of the match
you want separately, not just confirmed as present.

In [9]:
log_line = "2026-09-08 ERROR connection from 10.0.0.5"

match = re.search(r"(\d{4}-\d{2}-\d{2}) (\w+)", log_line)
print(match.group(0))   # the whole match: 2026-09-08 ERROR
print(match.group(1))   # just the date:   2026-09-08
print(match.group(2))   # just the level:  ERROR

2026-09-08 ERROR
2026-09-08
ERROR


Named groups — `(?P<name>...)` — read back much more clearly than counting
parentheses:

```python
match = re.search(r"(?P<date>\d{4}-\d{2}-\d{2}) (?P<level>\w+)", log_line)
print(match.group("date"))     # 2026-09-08
print(match.group("level"))    # ERROR
```

### Your turn
Using `log_line`, write a named-group pattern that also captures the IP address
(as `ip`), then print all three pieces by name.

In [10]:
pattern = (
    r"(?P<date>\d{4}-\d{2}-\d{2}) (?P<level>\w+) "
    r"connection from (?P<ip>\d+\.\d+\.\d+\.\d+)"
)
match = re.search(pattern, log_line)
if match:
    print(match.group("date"))
    print(match.group("level"))
    print(match.group("ip"))


2026-09-08
ERROR
10.0.0.5


---
## 6. `re.sub()` — find and replace by shape

Same idea as Week 7's `.replace()`, but matching a *shape* instead of exact
text — useful for redacting anything that looks like sensitive data.

In [11]:
text = "Server IP: 192.168.0.10, DNS: 8.8.8.8, Gateway: 10.0.0.1"

redacted = re.sub(r"\d+\.\d+\.\d+\.\d+", "[REDACTED]", text)
print(redacted)

Server IP: [REDACTED], DNS: [REDACTED], Gateway: [REDACTED]


### Your turn
Given `log_line` from section 5, use `re.sub()` to replace just the IP address
with `"***"`, leaving the date and level untouched.

In [12]:
redacted_line = re.sub(r"\d+\.\d+\.\d+\.\d+", "***", log_line)
print(redacted_line)


2026-09-08 ERROR connection from ***


---
## Topic 3 Challenge

Pull 5 repos from the GitHub API (Topic 1). For each one, check whether its
description mentions a version number shaped like `3.12`, using `re.search()`.

In [13]:
import requests

response = requests.get(
    "https://api.github.com/orgs/python/repos",
    params={"per_page": 5, "sort": "updated"},
    timeout=20,
)
response.raise_for_status()
repos = response.json()

for repo in repos:
    description = repo.get("description") or ""
    version = re.search(r"\b\d+\.\d+\b", description)
    if version:
        print(repo["name"], "version:", version.group())
    else:
        print(repo["name"], "no version number found")


/Users/mwave/Downloads/CST1510/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


cpython no version number found
mypy no version number found
typeshed no version number found
docs-community no version number found
packaging-council no version number found


---
## Same idea, three fields

| Field | Looks like |
|---|---|
| **AI / Data Science** | pulling every number out of a free-text survey answer |
| **Cyber Security** | pulling every IP address out of a raw log file |
| **IT** | pulling a version number or hostname pattern out of free-text config |

---
## Common mistakes

| Mistake | What you see | Fix |
|---|---|---|
| calling `.group()` without checking for `None` first | `AttributeError: 'NoneType' object has no attribute 'group'` | `if match:` before using it |
| using `re.search()` when you want every match | only the first one comes back | use `re.findall()` instead |
| forgetting the `r` before the pattern string | usually still works here, but backslashes can misbehave without it | always write `r"..."` for a regex pattern |
| a pattern that is too loose or too strict | matches the wrong thing, or nothing at all | test it on a few real examples before trusting it |
| `match.group(1)` on a pattern with no `(...)` | `IndexError: no such group` | `group(1)` needs a capture group in the pattern first |

## Go deeper

`examples/01_search_and_findall.py` — search, findall, the None case
`examples/02_regex_on_real_data.py` — applying a pattern to real pulled API data
`examples/03_groups_and_sub.py` — capture groups, named groups, and `re.sub()`
Docs: [re module](https://docs.python.org/3/library/re.html)

---
## Recap

| You can now | Using |
|---|---|
| find the first match of a shape | `re.search(pattern, text)` |
| find every match | `re.findall(pattern, text)` |
| check before using a result | `if match:` |
| match digits | `\d+` |
| pull out one piece of a match | `(...)` and `match.group(1)`, or `(?P<name>...)` |
| find and replace by shape | `re.sub(pattern, replacement, text)` |

**Next:** Topic 4 — turning what you have built into one class.